In [12]:
import numpy as np 
import pandas as pd 
import os
from datasets import load_dataset, Dataset
from sklearn.model_selection import GroupShuffleSplit 
from transformers import AutoTokenizer

#fixed seeds for reproducibility
random_state = 42

In [13]:
# load dataset from hugging face
data = load_dataset("b-mc2/sql-create-context")

#convert the data to a Pandas DataFrame
df = data["train"].to_pandas()

In [14]:
#splitting train/test/val without data leakage
#randomly divide data into subsets while keeping entire schema groups together
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state = random_state) #split to training and holdout(test and val)
train_idx, hold_idx = next(gss.split(df, groups=df["context"])) #the context column defines the groups
train_df, hold_df = df.iloc[train_idx], df.iloc[hold_idx] #convert arrays to dataframes 

#split the holdout 20% of groups in 2 : val and test
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.5, random_state = random_state)
val_idx, test_idx = next(gss2.split(hold_df, groups=hold_df["context"]))
val_df, test_df = hold_df.iloc[val_idx], hold_df.iloc[test_idx]

"""print(len(train_df), len(val_df), len(test_df))

print("Train/Validation schema overlap:",
      len(set(train_df["context"]) & set(val_df["context"])))

print("Train/Test schema overlap:",
      len(set(train_df["context"]) & set(test_df["context"])))

print("Validation/Test schema overlap:",
      len(set(val_df["context"]) & set(test_df["context"])))"""

'print(len(train_df), len(val_df), len(test_df))\n\nprint("Train/Validation schema overlap:",\n      len(set(train_df["context"]) & set(val_df["context"])))\n\nprint("Train/Test schema overlap:",\n      len(set(train_df["context"]) & set(test_df["context"])))\n\nprint("Validation/Test schema overlap:",\n      len(set(val_df["context"]) & set(test_df["context"])))'

In [15]:
#tokenization
tok = AutoTokenizer.from_pretrained("t5-small")

def make_input(q, c):
    return f"question: {q} context: {c}"

#test tokenization on the first row of training data
ex = train_df.iloc[0]
text = make_input(ex.question, ex.context)
ids = tok(text).input_ids #tokenized IDs
print(text)
print(ids)
print(tok.convert_ids_to_tokens(ids))

question: How many heads of the departments are older than 56 ? context: CREATE TABLE head (age INTEGER)
[822, 10, 571, 186, 7701, 13, 8, 10521, 33, 2749, 145, 11526, 3, 58, 2625, 10, 205, 4386, 6048, 332, 17098, 819, 41, 545, 3, 21342, 17966, 61, 1]
['▁question', ':', '▁How', '▁many', '▁heads', '▁of', '▁the', '▁departments', '▁are', '▁older', '▁than', '▁56', '▁', '?', '▁context', ':', '▁C', 'RE', 'ATE', '▁T', 'ABLE', '▁head', '▁(', 'age', '▁', 'INTE', 'GER', ')', '</s>']


In [16]:
#make sure the tokenizer can represent SQL characters and symbols
sample = train_df.sample(5000, random_state=0)
unk = tok.unk_token_id

all_chars = set("".join(sample.answer))
bad_chars = [ch for ch in all_chars if ch.strip() and unk in tok(ch).input_ids]
print("Characters the tokenizer can't represent:", bad_chars)

has_unk = sample.answer.map(lambda s: unk in tok(s).input_ids)
print(f"SQL answers affected: {has_unk.mean():.1%}")
print(sample.answer[has_unk].head(3).tolist())

Characters the tokenizer can't represent: ['√', 'に', 'ি', 'ñ', 'ก', '<', 'ו', '凰', 'ㅎ', 'ら', '粉', 'ҭ', '¥', 'ų', '沒', 'ч', '金', '兒', '†', 'ś', 'ֹ', 'ן', '成', 'ш', 'ิ', 'О', 'า', 'ങ', '九', '必', 'ย', 'ม', '↑', 'ব', 'ɡ', '密', 'ָ', 'М', 'ี', '视', 'ҕ', 'ø', '‡', 'ʼ', '有', '്', 'š', '៨', 'ച', 'å', '的', 'ğ', '痛', 'র', '牌', '☪', '州', '第', '×', '卫', 'י', '配', '会', 'ŋ', 'ь', '功', 'ค', 'ë', 'น', 'の', '方', '呼', 'ė', '~', 'ć', 'ち', 'β', 'נ', '凤', 'ń', 'ā', 'ò', 'ə', '訴', '夜', '宜', 'ം', 'Б', 'ח', 'ō', '台', '僕', '电', '～', 'ә', 'お', '秘', 'ɣ', 'ą', '乳', 'た', 'ž', '២', 'ി', 'ɪ', '½', 'Ҭ', '歌', '告', '你', '都', '恋', '幼', '吸', 'П', '嬰', '·', 'í', '良', 'ł', '影', '−', '氏', 'α', 'ั', 'г', 'ʁ', 'া', 'ū', 'ɐ', '情', 'ã']
SQL answers affected: 10.5%
['SELECT COUNT(series__number) FROM table_name_20 WHERE season__number < 22 AND directed_by = "matthew penn"', 'SELECT type FROM table_2562572_33 WHERE cyrillic_name_other_names = "Ором (Hungarian: Orom)"', 'SELECT nation FROM table_name_28 WHERE gold = 0 AND bronze < 

In [17]:
#measure the tokens in the model's input/output to choose max sequence length
lens_in  = [len(tok(make_input(q, c)).input_ids) for q, c in zip(sample.question, sample.context)]
lens_out = [len(tok(a).input_ids) for a in sample.answer]
for name, l in [("input", lens_in), ("output", lens_out)]:
    print(name, "median / 95% / 99% / max:", np.percentile(l, [50, 95, 99, 100]).astype(int))

input median / 95% / 99% / max: [ 46  74  90 148]
output median / 95% / 99% / max: [ 29  48  72 191]


In [18]:
#func to tokenize a batch of text and detect which ones contain unknown tokens
unk = tok.unk_token_id

def has_unk(texts):
    ids = tok(list(texts), add_special_tokens=False)["input_ids"]
    return np.array([unk in x for x in ids])

In [19]:
#replace important characters that produce unknown tokens with placeholders
PH = {
    "<": "LESSTHAN",
    "~": "TILDESYM",
    "{": "LBRACESYM",
    "}": "RBRACESYM",
    "\\": "BACKSLASHSYM",
    "`": "BACKTICKSYM",
}

def encode_text(s):
    for ch, p in PH.items():
        s = s.replace(ch, p)
    return s

def decode_text(s):
    for ch, p in PH.items():
        s = s.replace(p, ch)
    return s

# 1) No placeholder may already exist in the data
for name, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    for col in ["question", "context", "answer"]:
        for p in PH.values():
            assert not d[col].str.contains(p, regex=False).any(), f"{p} in {name}.{col}"

# 2) Round trip must be exact
for d in (train_df, val_df, test_df):
    assert (d.answer.map(encode_text).map(decode_text) == d.answer).all()
print("Placeholders safe, round trip OK")

# 3) Any ASCII gaps left?
chars = set("".join(train_df.answer.map(encode_text)))
print("Unrepresentable ASCII:", [c for c in chars if c.isascii() and c.strip()
      and unk in tok(c, add_special_tokens=False).input_ids])

Placeholders safe, round trip OK
Unrepresentable ASCII: []


In [20]:
# prepare train, test, val datasets
def build(d):
    d = d.copy()
    d["input_text"]  = [encode_text(f"question: {q} context: {c}") for q, c in zip(d.question, d.context)]
    d["target_text"] = d.answer.map(encode_text)
    d["has_unk"] = has_unk(d.input_text) | has_unk(d.target_text)
    return d.reset_index(drop=True)

train_b, val_b, test_b = build(train_df), build(val_df), build(test_df)
train_b = train_b[~train_b.has_unk].reset_index(drop=True)   # train only!
print(len(train_b), len(val_b), len(test_b))
print("clean fraction of test:", 1 - test_b.has_unk.mean())

61797 7859 7823
clean fraction of test: 0.9815927393583025


In [21]:
#tokenize datasets and apply max sequence length
MAX_IN, MAX_OUT = 192, 192

def tokenize(batch):
    enc = tok(batch["input_text"], max_length=MAX_IN, truncation=True)
    enc["labels"] = tok(text_target=batch["target_text"],
                        max_length=MAX_OUT, truncation=True)["input_ids"]
    return enc

def to_ds(d):
    ds_ = Dataset.from_pandas(d[["input_text", "target_text"]])
    return ds_.map(tokenize, batched=True, remove_columns=["input_text", "target_text"])

train_ds, val_ds, test_ds = to_ds(train_b), to_ds(val_b), to_ds(test_b)

print("Truncated inputs :", sum(len(x) == MAX_IN  for x in train_ds["input_ids"]))
print("Truncated targets:", sum(len(x) == MAX_OUT for x in train_ds["labels"]))
print(tok.decode(train_ds[0]["labels"], skip_special_tokens=True))

Map:   0%|          | 0/61797 [00:00<?, ? examples/s]

Map:   0%|          | 0/7859 [00:00<?, ? examples/s]

Map:   0%|          | 0/7823 [00:00<?, ? examples/s]

Truncated inputs : 3
Truncated targets: 5
SELECT COUNT(*) FROM head WHERE age > 56
